# 01 Discovery, ground truth and incident analysis (slice 1)

Goal: find the DNS tunnel in `dns.log`, confirm it in `case1.pcap`, label every row as `benign` or `tunnel`, and save `data/labels.csv`. Sections 7 to 12 then reconstruct what happened inside the tunnel (report 2.2) and measure how it differs from normal DNS (report 2.3).

**Row id convention (shared with the group):** `row_id` is the 0-based position of the row among the data rows of `dns.log` (header lines starting with `#` skipped, file order kept).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
if not (RAW / "dns.log").exists():  # Canvas download keeps the files in a "Case 1" folder
    RAW = RAW / "Case 1"

# Zeek names the columns itself: the line starting with #fields lists them in order
with open(RAW / "dns.log", encoding="latin-1") as f:
    COLS = next(line for line in f if line.startswith("#fields")).rstrip("\n").split("\t")[1:]

df = pd.read_csv(RAW / "dns.log", sep="\t", comment="#", names=COLS,
                 na_values=["-"], keep_default_na=False, encoding="latin-1")
df.index.name = "row_id"
df["dt"] = pd.to_datetime(df["ts"], unit="s")
df["qlen"] = df["query"].str.len()
print(len(COLS), "columns read from the #fields line")
print(df.shape, df["dt"].min(), "to", df["dt"].max())

## 1. First look: who talks, with what

In [ ]:
display(df["id.orig_h"].value_counts().to_frame("queries"))
display(df["qtype_name"].value_counts().to_frame("queries"))
display(df["rcode_name"].value_counts(dropna=False).to_frame("queries"))

## 2. Rank parent domains: which one does not look like the others?

In [ ]:
df["parent"] = df["query"].str.split(".").str[-2:].str.join(".")
parents = (df.groupby("parent")
             .agg(queries=("query", "size"), unique_names=("query", "nunique"),
                  mean_len=("qlen", "mean"), max_len=("qlen", "max"),
                  clients=("id.orig_h", "nunique"))
             .sort_values("unique_names", ascending=False))
parents.head(10)

`pirate.sea` has by far the most unique names and by far the longest names. Next: is it also one client and one record type?

In [ ]:
is_domain = df["query"].str.endswith(".pirate.sea")
is_null = df["qtype_name"] == "NULL"
is_host = df["id.orig_h"] == "10.0.2.30"
display(pd.crosstab([is_domain.rename("pirate.sea"), is_null.rename("qtype NULL")], is_host.rename("client 10.0.2.30")))
print("longest benign name:", df.loc[~is_domain, "qlen"].max(), "chars; longest tunnel name:", df.loc[is_domain, "qlen"].max())

## 3. Timeline of the tunnel

In [ ]:
t = df[is_domain]
print("first:", t["dt"].min(), " last:", t["dt"].max(), " rows:", len(t), " unique names:", t["query"].nunique())
print("rows without response (rcode empty):", t["rcode_name"].isna().sum())
display(t.groupby(t["dt"].dt.floor("5min")).size().to_frame("tunnel queries"))

fig, ax = plt.subplots(figsize=(10, 3))
per_min = df.groupby([df["dt"].dt.floor("1min"), is_domain.rename("tunnel")]).size().unstack(fill_value=0)
per_min.plot(ax=ax, kind="bar", stacked=True, width=1.0, legend=True)
ax.set_xticks(ax.get_xticks()[::10]); ax.set_ylabel("queries per minute"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Is the second burst a repeat of the first? Compare the names sent in each burst.
split = pd.Timestamp("2023-01-07 18:30:00")
first, second = set(t.loc[t["dt"] < split, "query"]), set(t.loc[t["dt"] >= split, "query"])
print("burst 1:", len(first), "names | burst 2:", len(second), "names | shared:", len(first & second))
t["query"].head(5).tolist()

## 4. Confirm in the PCAP

`dns.log` is Zeek's view. The PCAP is the ground truth for what was on the wire. We check that the packet-level count of DNS queries matches the log and that the `pirate.sea` queries are NULL queries from `10.0.2.30`.

In [ ]:
from scapy.all import rdpcap, DNS, DNSQR, IP
pkts = rdpcap(str(RAW / "case1.pcap"))
print("packets in pcap:", len(pkts))

rows = []
for p in pkts:
    if p.haslayer(DNS) and p.haslayer(IP):
        d = p[DNS]
        qname = d.qd.qname.decode("latin-1").rstrip(".") if d.qd else None
        rows.append((float(p.time), p[IP].src, p[IP].dst, d.qr, d.qd.qtype if d.qd else None, qname, len(p)))
pc = pd.DataFrame(rows, columns=["ts", "src", "dst", "is_response", "qtype", "qname", "frame_len"])
pc["dt"] = pd.to_datetime(pc["ts"], unit="s")
print("DNS packets:", len(pc), "| queries:", (pc["is_response"] == 0).sum(), "| responses:", (pc["is_response"] == 1).sum())
pc_t = pc[pc["qname"].fillna("").str.endswith(".pirate.sea")]
print("pirate.sea packets:", len(pc_t), "| qtype values:", pc_t["qtype"].unique(), "| sources of queries:", pc_t.loc[pc_t.is_response == 0, "src"].unique())
display(pc_t.groupby("is_response")["frame_len"].describe())

## 5. Label and save

**Rule:** `tunnel` = query name under `pirate.sea`. Everything else is `benign`. The rule agrees with the client (`10.0.2.30`) and the record type (NULL) criteria on every row, shown in section 2.

In [ ]:
labels = pd.DataFrame({"label": np.where(is_domain, "tunnel", "benign")}, index=df.index)
assert (is_domain == is_null).all() and (is_domain == is_host).all(), "criteria disagree, inspect before labelling"
labels.to_csv(ROOT / "data" / "labels.csv")
print(labels["label"].value_counts())
print("tunnel share: %.1f%%" % (100 * (labels["label"] == "tunnel").mean()))

## 6. Sanity checks on the saved file

In [ ]:
chk = pd.read_csv(ROOT / "data" / "labels.csv", index_col="row_id")
assert len(chk) == len(df) and chk.index.equals(df.index)
print("rows match dns.log:", len(chk))
print("random benign sample (read by eye):")
display(df.loc[chk.index[chk["label"] == "benign"], ["id.orig_h", "query", "qtype_name"]].sample(15, random_state=1))
print("benign rows that look odd (NXDOMAIN or long name):")
display(df.loc[(chk["label"] == "benign") & ((df["rcode_name"] == "NXDOMAIN") | (df["qlen"] > 50)),
               ["id.orig_h", "query", "qtype_name", "rcode_name"]])

## 7. Pair every tunnel query with its answer

What travelled through the tunnel is only in the PCAP: Zeek logs every answer as `<unknown type=10>`. We put each `pirate.sea` query in one row together with its answer. Answers are matched on the DNS transaction ID within a burst, because burst 2 reuses the IDs of burst 1 (section 9).

In [ ]:
import re, zlib
from collections import Counter
from scapy.all import Ether, UDP, TCP, DNSRROPT
from scapy.layers.dns import dnstypes

rows = []
for frame, p in enumerate(pkts, start=1):
    if not (p.haslayer(DNS) and p[DNS].qd.qname.endswith(b".pirate.sea.")):
        continue
    d = p[DNS]
    rows.append(dict(frame=frame, dt=pd.to_datetime(float(p.time), unit="s"), qr=d.qr, tid=d.id,
                     name=d.qd.qname[:-len(".pirate.sea.")],   # raw bytes: the part the tunnel controls
                     dns_len=len(d), edns0=any(isinstance(r, DNSRROPT) for r in d.ar),
                     rdata=bytes(d.an[0].rdata) if d.ancount else None))
tp = pd.DataFrame(rows)
tp["burst"] = np.where(tp["dt"] < split, 1, 2)

s = (tp[tp.qr == 0].drop(columns=["qr", "rdata"])
       .merge(tp.loc[tp.qr == 1, ["burst", "tid", "dt", "dns_len", "rdata"]],
              on=["burst", "tid"], how="left", suffixes=("", "_ans"))
       .sort_values("dt", ignore_index=True))
s["prefix"] = s["name"].map(lambda n: chr(n[0]))
print(len(s), "tunnel queries,", s["dt_ans"].notna().sum(), "answered")
s.head()

## 8. Is 10.0.2.20 a resolver or the tunnel server?

A recursive resolver answers with RA=1 ("recursion available") and copies RD from the query. iodine's server instead hard-codes AA=1, RD=0, RA=0 and TTL 0 in every answer (`src/dns.c` in the iodine repository). We compare the answer flags of both DNS servers in the capture. We also check whether a router sits between client and server: a router would rewrite the MAC addresses and lower the IP TTL.

In [ ]:
answers = pd.DataFrame([(p[IP].src, p[DNS].aa, p[DNS].rd, p[DNS].ra,
                         "none" if not p[DNS].ancount else "0" if p[DNS].an[0].ttl == 0 else "> 0")
                        for p in pkts if p.haslayer(DNS) and p[DNS].qr == 1],
                       columns=["server", "AA", "RD", "RA", "answer TTL"])
display(answers.value_counts().to_frame("answers"))

links = pd.DataFrame([(p[IP].src, p[Ether].src, p[Ether].dst, p[IP].ttl) for p in pkts if p.haslayer(DNS)],
                     columns=["sender", "sender MAC", "receiver MAC", "IP TTL"])
links.value_counts().to_frame("packets")

All 424 tunnel answers carry iodine's fixed header, while the home router (192.168.179.1) answers like a normal resolver. The client sends its frames straight to the server's MAC address and the IP TTL is 64 in both directions, so client and server sit on the same network segment. **10.0.2.20 is the iodine server itself, and in this lab no network perimeter is crossed.** The MAC prefix `08:00:27` is VirtualBox; the benign pair is an Apple device (`14:7d:da`) behind an AVM FRITZ!Box (`dc:15:c8`), so the capture combines two separate networks.

## 9. Is burst 2 a second session or a copy of burst 1?

Each new iodine session gets a fresh random login challenge from the server, so its answers differ from any earlier session. If burst 2 has the same names, the same IDs and byte-identical answers, it is a copy of the recorded session, not a new one.

In [ ]:
b1, b2 = (s[s.burst == b].reset_index(drop=True) for b in (1, 2))
print("same names, same order:", b1["name"].equals(b2["name"]))
print("same transaction IDs:  ", b1["tid"].equals(b2["tid"]))
print("identical answers:     ", (b1["rdata"] == b2["rdata"]).sum(), "of", b1["rdata"].notna().sum())
print("login challenge 1 / 2: ", b1.loc[0, "rdata"][4:8].hex(), "/", b2.loc[0, "rdata"][4:8].hex())
shift = (b2["dt"] - b1["dt"]).dt.total_seconds()
print(f"burst 2 starts {shift[0]:.1f} s later; its relative timing differs by at most {(shift - shift[0]).abs().max():.2f} s")

Both directions are identical, including the random challenge, and the timing matches within 0.3 s. Burst 2 is most likely the recorded session played back a second time when the dataset was built. **The data holds one session, not two.** From here on we only analyse burst 1.

## 10. Timeline of the session

The first character of every query name is an iodine command (protocol specification `doc/proto_00000502.txt`). Grouping on it gives the steps of the session in order. Most handshake answers are plain text, so we can read them directly.

In [ ]:
STEPS = {"v": "version", "l": "login", "y": "downstream codec check", "z": "upstream codec check",
         "s": "switch upstream codec", "o": "options", "r": "fragment size probe",
         "n": "set fragment size", "p": "ping (poll)", "1": "data, user 1"}
b1["step"] = b1["prefix"].map(STEPS)
display(b1.groupby("step", sort=False).agg(queries=("tid", "size"), answered=("dt_ans", "count"),
                                           first=("dt", "min"), last=("dt", "max")))
for i in b1.index[b1["prefix"].isin(list("vlson"))]:
    print(f"{b1.loc[i, 'step']:22} answer: {b1.loc[i, 'rdata']}")

How to read the answers: `VACK` + a 4-byte login challenge + user ID 1; the login reply gives the server's and client's addresses inside the tunnel (10.20.30.1 and 10.20.30.3), the tunnel MTU (1,130 bytes) and the netmask (/24); the client switches its uploads to Base128 and turns on lazy mode; the fragment size is set to `0x04a2` = 1,186 bytes.

iodine encodes the names with its own alphabets (Base32 `a-z0-5`, Base128 `a-zA-Z0-9` plus bytes `0xBC-0xFD`), most significant bit first. One small function decodes both. We use it to read the protocol version in the first query, and the sizes the client asked for while probing.

In [ ]:
B32 = b"abcdefghijklmnopqrstuvwxyz012345"
B128 = b"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789" + bytes(range(0xBC, 0xFE))

def decode(text, alphabet, bits):
    # every character carries `bits` bits; glue them together and cut into bytes
    stream = "".join(format(alphabet.index(c), f"0{bits}b") for c in text)
    return bytes(int(stream[i:i + 8], 2) for i in range(0, len(stream) - 7, 8))

print("protocol version:", decode(b1.loc[0, "name"][1:], B32, 5)[:4].hex())

probes = b1[b1.prefix == "r"].copy()   # 3 Base32 characters: 4 bits user ID, 11 bits requested size
probes["requested_bytes"] = probes["name"].map(lambda n: (B32.index(n[1]) & 1) << 10 | B32.index(n[2]) << 5 | B32.index(n[3]))
probes[["dt", "requested_bytes", "dt_ans"]]

Version `00000502` is the protocol of the published iodine specification. The probing explains the 9-second gap in the handshake: answers of 1,344, 1,248 and 1,200 bytes never arrived, and each was tried three times with a 1-second timeout before the client settled on a smaller size.

## 11. What did the tunnel carry?

Following the specification:
- **Upstream** (data queries, name starts with user ID `1`): 5 header characters (user ID; 3 Base32 characters holding sequence number, fragment number and a last-fragment flag; a cache-miss counter), then the payload in Base128.
- **Downstream** (answers to pings and data queries): a 2-byte header (compression flag in byte 1; fragment number and last-fragment flag in byte 2), then raw bytes.

Fragments are joined until the last-fragment flag and decompressed with zlib. The result is an IP packet with a 4-byte tunnel header in front, which scapy can read.

In [ ]:
def reassemble(fragments):
    # fragments: (time, fragment number, bytes, is_last, compressed) in arrival order
    packets, buf = [], b""
    for t, frag, data, last, compressed in fragments:
        buf = data if frag == 0 else buf + data
        if last:
            raw = zlib.decompress(buf) if compressed else buf
            packets.append((t, IP(raw[4:])))   # skip the 4-byte tunnel header
    return packets

up_frags = []
for t, name in b1.loc[b1.prefix == "1", ["dt", "name"]].itertuples(index=False):
    name = name.replace(b".", b"")
    h = [B32.index(c) for c in name[1:4]]
    up_frags.append((t, (h[0] & 3) << 2 | h[1] >> 3, decode(name[5:], B128, 7), h[2] & 1, True))

down = b1[b1.prefix.isin(["p", "1"]) & (b1["rdata"].map(lambda r: len(r) if isinstance(r, bytes) else 0) > 2)].sort_values("dt_ans")
down_frags = [(t, r[1] >> 1 & 15, r[2:], r[1] & 1, r[0] >> 7) for t, r in zip(down["dt_ans"], down["rdata"])]

inner = pd.DataFrame([(t, "up", pkt) for t, pkt in reassemble(up_frags)] +
                     [(t, "down", pkt) for t, pkt in reassemble(down_frags)],
                     columns=["dt", "dir", "pkt"]).sort_values("dt", ignore_index=True)
inner["flow"] = inner["pkt"].map(lambda p: f"{p.sprintf('%IP.proto%')} {p[IP].src}:{p.sport} > {p[IP].dst}:{p.dport}")
inner["ip_bytes"] = inner["pkt"].map(len)
inner.groupby(["dir", "flow"]).agg(packets=("pkt", "size"), ip_bytes=("ip_bytes", "sum"),
                                   first=("dt", "min"), last=("dt", "max"))

Two flows: TCP to port 1720 and UDP on port 5000. These are the ports of an **H.323 voice call**: call set-up as Q.931 messages over TCP (each behind a 4-byte TPKT header), and the audio as RTP over UDP. We read the call set-up messages and the readable text inside them.

In [ ]:
Q931 = {0x05: "SETUP", 0x02: "CALL PROCEEDING", 0x07: "CONNECT", 0x62: "FACILITY", 0x5A: "RELEASE COMPLETE"}
signalling = inner[inner.pkt.map(lambda p: bool(p.haslayer(TCP)) and len(p[TCP].payload) > 0)]
for direction, grp in signalling.groupby("dir"):
    stream, starts = b"", []
    for t, p in zip(grp["dt"], grp["pkt"]):
        starts.append((len(stream), t))
        stream += bytes(p[TCP].payload)
    pos = 0
    while pos < len(stream):                      # TPKT: version 3, reserved, 2-byte length
        n = int.from_bytes(stream[pos + 2:pos + 4], "big")
        msg = stream[pos + 4:pos + n]
        kind = Q931.get(msg[2 + msg[1]], "?")     # message type follows the call reference
        cause = f" cause {msg[6 + msg[1]] & 0x7F}" if kind == "RELEASE COMPLETE" else ""
        texts = [x.decode() for x in re.findall(rb"[A-Za-z][ -~]{3,}", msg)][:3]
        print(max(t for o, t in starts if o <= pos).time(), f"{direction:5} {kind}{cause}", texts)
        pos += n

In [ ]:
rtp = inner[inner.pkt.map(lambda p: bool(p.haslayer(UDP)))]
for direction, grp in rtp.groupby("dir"):
    payload = grp["pkt"].map(lambda p: bytes(p[UDP].payload))
    print(f"{direction:5} {len(grp)} RTP packets, payload type {payload.map(lambda b: b[1] & 0x7F).unique()}, "
          f"{(grp.dt.max() - grp.dt.min()).total_seconds():.2f} s, {payload.map(lambda b: b[12:]).nunique()} distinct payloads")

audio = rtp[rtp.dir == "down"]
pings = b1[b1.prefix == "p"]
print(pings["dt_ans"].between(audio.dt.min(), audio.dt.max()).sum(), "of", len(pings),
      "ping answers arrive while the audio plays")
print("IP bytes carried: up", inner.loc[inner.dir == "up", "ip_bytes"].sum(),
      "| down", inner.loc[inner.dir == "down", "ip_bytes"].sum(),
      "| DNS bytes used:", int(b1["dns_len"].sum() + b1["dns_len_ans"].sum()))

The client (inside the tunnel 10.20.30.3, calling as "root" with SimpleOPAL) called an *OpenH323 Answering Machine* at the server's tunnel address 10.20.30.1. RTP payload type 3 is GSM audio: one 33-byte frame per 20 ms, so 169 frames are about 3.4 seconds of audio, mostly distinct, so a real recording. The answering machine then ended the call (cause 16 is "normal call clearing"). Each downstream audio frame arrived as the answer to a ping, which is why the pings come every 20 ms. Upstream carried only the call set-up and two audio frames: **no file transfer or other exfiltration is visible.**

## 12. How the tunnel differs from normal DNS (input for 2.3)

For each candidate characteristic we compare tunnel and benign traffic in the whole PCAP. Name lengths are bytes of the name as sent, without Zeek's `\xNN` escapes. Entropy is the Shannon entropy of the part left of the parent domain, in bits per character.

In [ ]:
def entropy(b):
    if not b:
        return 0.0
    p = np.array(list(Counter(b).values())) / len(b)
    return float(-(p * np.log2(p)).sum())

rows = []
for p in pkts:
    if p.haslayer(DNS):
        d, name = p[DNS], p[DNS].qd.qname.rstrip(b".")
        labels = name.split(b".")
        rows.append(dict(side="tunnel" if name.endswith(b"pirate.sea") else "benign", qr=d.qr,
                         dt=pd.to_datetime(float(p.time), unit="s"), name=name, parent=b".".join(labels[-2:]),
                         name_bytes=len(name), longest_label=max(map(len, labels)),
                         entropy=entropy(b".".join(labels[:-2])), qtype=dnstypes.get(d.qd.qtype),
                         edns0=any(isinstance(r, DNSRROPT) for r in d.ar), dns_len=len(d), rcode=d.rcode,
                         sport=p[UDP].sport))
dq = pd.DataFrame(rows)
qs, rs = dq[dq.qr == 0].groupby("side"), dq[dq.qr == 1].groupby("side")

pd.DataFrame({
    "name length, median / max (bytes)": qs["name_bytes"].agg(lambda x: f"{x.median():.0f} / {x.max()}"),
    "longest label, max (bytes)":        qs["longest_label"].max(),
    "entropy, median (bits/char)":       qs["entropy"].median().round(2),
    "unique names, busiest parent":      dq[dq.qr == 0].groupby(["side", "parent"])["name"].nunique().groupby("side").max(),
    "record types":                      qs["qtype"].agg(lambda x: ", ".join(x.value_counts().index)),
    "queries with EDNS0 (%)":            (qs["edns0"].mean() * 100).round(1),
    "answer size, median / max (bytes)": rs["dns_len"].agg(lambda x: f"{x.median():.0f} / {x.max()}"),
    "NXDOMAIN answers":                  rs["rcode"].agg(lambda x: int((x == 3).sum())),
    "peak queries per minute":           dq[dq.qr == 0].groupby(["side", dq["dt"].dt.floor("1min")]).size().groupby("side").max(),
    "source ports":                      qs["sport"].nunique(),
}).T

Not every step of the session looks the same, which matters for the "stage" question in 2.3. Per step: how long and how random the names are, and how big the answers are.

In [ ]:
(b1.assign(name_bytes=b1["name"].map(len) + len(".pirate.sea"), entropy=b1["name"].map(entropy))
   .groupby("step", sort=False)
   .agg(queries=("tid", "size"), name_bytes=("name_bytes", "median"),
        entropy=("entropy", "median"), answer_bytes=("dns_len_ans", "median"))
   .round(2))

**Takeaway for 2.3.** Every tunnel query shares one parent domain with hundreds of unique names, uses NULL and carries EDNS0. Long, random names only appear in the probe and data queries; the 173 pings are as short as normal names, so a per-query length or entropy rule misses most of the session. Query volume (the benign client is busier) and NXDOMAIN (none in the tunnel) do not separate the two. The single source port is an indicator for this session, not a characteristic: any client can change it.

## Summary: what we did and why

1. **Loaded `dns.log`** (3,645 DNS queries). Column names come from the log's own `#fields` line.
2. **Looked at who talks and with what.** Two clients on two separate networks, each with its own DNS server (for the tunnel client that server is the iodine server, section 8). One client sends only NULL records.
3. **Ranked parent domains.** `pirate.sea` stands out on unique names (222), name length and record type. It does not stand out on query volume.
4. **Cross-checked the suspicion.** Domain, record type NULL and client `10.0.2.30` agree on every row, so one simple rule labels everything without conflicts.
5. **Plotted the timeline.** Two bursts of about 25 seconds, 46 minutes apart (18:04 and 18:50 to 18:51). The second burst repeats the first exactly (section 9).
6. **Confirmed in the PCAP.** The log is complete (3,645 queries, 21 unanswered). The PCAP also holds the answer payload that Zeek cannot decode.
7. **Labelled and saved** `data/labels.csv`: 444 tunnel, 3,201 benign (12.2% tunnel). The rule is "the name ends in `pirate.sea`".
8. **Sanity-checked the labels.** Row counts match, a random benign sample looks like normal browsing, and the odd-looking benign rows (long cloud names, NXDOMAIN) are not tunnel traffic. They are the likely false positives later.
9. **Paired every tunnel query with its answer** from the PCAP, because Zeek does not log the answer payload.
10. **Identified the server.** 10.0.2.20 answers with iodine's fixed flags (AA=1, RA=0, TTL 0), so it is the iodine server, not a resolver. Client and server share one network segment, so no perimeter is crossed in this lab.
11. **Checked the second burst.** It is a byte-identical copy of the first, including the random login challenge: one session, recorded twice.
12. **Rebuilt the timeline** from the command letter of each query: version, login, encoding tests, fragment-size probing (9 s lost on failed probes), then data and polling.
13. **Decoded the tunnel.** It carried one H.323 voice call from "root" to an OpenH323 answering machine: about 3.4 s of GSM audio downstream, call set-up upstream, no exfiltration.
14. **Compared tunnel and benign traffic** (input for 2.3): unique names under one domain, NULL records, EDNS0 and large answers separate the tunnel; long, high-entropy names only mark the probe and data queries; query volume and NXDOMAIN do not separate at all.

**Indicators for Appendix A:** domain `pirate.sea`; client 10.0.2.30 (`08:00:27:9c:e0:b4`, source port 44639); iodine server 10.0.2.20 (`08:00:27:c7:6e:ba`); tunnel addresses 10.20.30.1 and 10.20.30.3; iodine protocol `00000502`; H.323 software "SimpleOPAL 3.6.8" and "OpenH323 Answering Machine v1.1.18".